# 模型量化（Quantization）
当我们在显存受限的边缘设备、单张消费级显卡（如 RTX 3090/4090）或私有云服务器上部署数十亿至数百亿参数的大模型时，量化是将显存占用压缩 50%~75% 且基本保持模型精度无损的必由之路。

## AWQ 与 GPTQPost-Training 量化算法原理
1. 深入理解 PTQ（Post-Training Quantization）训练后量化：掌握权重量化（Weight-Only Quantization）的数学原理。
2. 拆解两大主流量化算法（GPTQ vs AWQ）：理清基于二阶梯度的矩阵补偿（GPTQ）与基于激活值特征感知的保护量化（AWQ）的区别
3. 使用 AutoAWQ/vLLM 直接加载 AWQ/GPTQ 4-bit 量化模型并进行高效推理。

## GPTQ 与 AWQ 量化算法硬核对比
量化是将浮点数权重（如 FP16，16 bit）映射到低精度整数（如 INT4，4 bit）的过程。最基础的线性量化容易导致模型在边缘离群值（Outliers）上损失大量精度。为此，学术界与工业界推出了两种极其出色的 Post-Training Quantization (PTQ) 算法：

[原始 FP16 权重]
       ├── GPTQ: 基于二阶 Hessian 矩阵逐列量化并调整剩余权重补偿误差
       └── AWQ: 观察激活值分布，挑选 1% 的“显著权重”施加保护，仅量化其余 99%


#### GPTQ (Generalized Post-Training Quantization)
* 核心机制：采用逐列（Column-by-column）量化的策略。当把某一列权重从 FP16 强制压缩为 INT4 时，必然会产生量化误差 $\Delta W$。GPTQ 利用 Hessian 矩阵（二阶导数信息） 动态计算并修改该层中尚未量化的其余列权重，从而在全局上弥补这一列造成的精度损失。
* 特点：量化速度极快，适合处理超大规模模型（如 70B+）；但在 INT3 或极低比特下略容易失真。

#### AWQ (Activation-aware Weight Quantization)
* 核心机制：AWQ 发现并不是所有权重都同等重要——只有不到 1% 的权重（对应激活值幅度较大的特征通道）决定了模型的表达能力。
* 防失真保护：AWQ 不去强行对所有权重一视同仁地量化，而是通过校准集观察激活值（Activation），寻找出那 1% 的“显著权重（Salient Weights）”，对其进行缩放保护后再进行 INT4 量化。
* 特点：在 4-bit 量化下，AWQ 的困惑度（Perplexity）与推理精度显著优于传统量化，是目前 vLLM / TensorRT-LLM 部署 INT4 模型时的最首选推荐格式。

#### 使用 vLLM 直接加载与运行 AWQ 4-bit 量化模型
在生产环境中，我们通常无需自己从零运行耗时的量化校准过程，因为社区（如 Hugging Face Hub）已经提供了大量基于 AWQ/GPTQ 量化好的优质模型。vLLM 内部集成了极速的 AWQ / GPTQ INT4 CUDA 算子，可直接实现显存立减 70%、吞吐量大增的效果。
> 环境准备提示：运行本段代码需要安装 vLLM 与 autoawq 库：
`pip install vllm autoawq`

In [ ]:
from vllm import LLM, SamplingParams

# ================= 1. 指定预量化好的 AWQ 模型 ID =================
# 以 Qwen2.5-7B-Instruct-AWQ 为例 (仅需约 5GB 显存即可完美运行)
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ"

print("🚀 正在通过 vLLM 加载 4-bit AWQ 量化模型...")

# ================= 2. 初始化 vLLM 引擎，指定 quantization="awq" =================
llm = LLM(
    model=MODEL_ID,
    quantization="awq",      # 显式指定量化类型为 awq (vLLM 会自动调用极速 INT4 CUDA 算子)
    dtype="auto",            # 自动匹配计算精度 (通常为 float16/bfloat16)
    gpu_memory_utilization=0.8,
    max_model_len=2048
)

# ================= 3. 配置 Sampling 采样参数 =================
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=256
)

# ================= 4. 批量推理测试 =================
prompts = [
    "请用简短的语言对比 GPTQ 与 AWQ 量化算法的核心区别。",
    "写一段 Python 脚本，展示如何使用 vLLM 加载 AWQ 模型。"
]

print("\n🔥 开始 AWQ 量化模型的高速批量推理...")
outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"\n全量 Prompt: {prompt}")
    print(f"🤖 [AWQ Model Output]:\n{generated_text.strip()}")


**核心工程细节解析**
1. 显存占用大幅压缩：
    * 7B 模型（FP16/BF16）：权重需占用约 14 GB 显存。
    * 7B 模型（AWQ INT4）：权重仅占用约 4.5 GB ~ 5 GB 显存！即使在单张 RTX 3060/4060（8G 显存）的消费级显卡上，也能流畅部署并留出充足的显存空间给 KV Cache。

2. 计算过程中的反量化（De-quantization）：
    * 在 AWQ INT4 推理过程中，权重以 4-bit 形式保存在显存中；当数据流过 GPU Tensor Core 时，内部的高效 CUDA 算子会实时将 INT4 权重反量化为 FP16/BF16 进行矩阵乘法计算，从而在保持高吞吐的同时确保计算精度。

**思考**
1. W4A16 vs W4A4：我们在代码中使用的是 W4A16（权重 4-bit 量化，激活值 16-bit 保持高精度）模式。如果将激活值也强行量化为 4-bit（即 W4A4），可能会带来什么工程问题？
2. 量化对 Agent 工具调用的影响：将模型从 FP16 压缩至 INT4 量化后，模型的通用语言生成能力变化不大，但严格格式输出（如 JSON Schema / Structured Tool Calling）的成功率可能会受到什么程度的波动？在生产上应该如何测试评估？